# Survey Data Cleaning

This notebook processes the CS alumni and Alumni survey data to identify underserved populations.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np

## Load alumni Survey Data

In [2]:
# Load the alumni survey data
alumni_file = 'CS Alumni Survey (Co-SEA)_January 29, 2026_11.44 2.xlsx'
df_alumni = pd.read_excel(alumni_file, header=0)

print(f"Shape of alumni data: {df_alumni.shape}")
print(f"\nColumn names (first 10):")
print(df_alumni.columns.tolist()[:10])
print(f"\nFirst row (question text row):")
print(df_alumni.iloc[0][['Q29']].tolist() if 'Q29' in df_alumni.columns else 'Q29 not found')
print(f"\nActual data starts from row index 1")
print(f"Number of actual responses: {len(df_alumni) - 1}")

Shape of alumni data: (41, 51)

Column names (first 10):
['StartDate', 'EndDate', 'Status', 'IPAddress', 'Progress', 'Duration (in seconds)', 'Finished', 'RecordedDate', 'ResponseId', 'RecipientLastName']

First row (question text row):
['What is your race? (Please select all that apply)']

Actual data starts from row index 1
Number of actual responses: 40


## Analyze Race Column

Explore the race column to see all the different race options selected by respondents. Since this is a "select all that apply" question, people may have multiple races listed.

In [3]:
# Q29 is the race column
race_col = 'Q29'

def parse_races(race_entry):
    """
    Parse a race entry, splitting on commas but respecting parentheses.
    Example: "Black or African American,East Asian (e.g., Chinese, Japanese, Korean)"
    Should split into: ["Black or African American", "East Asian (e.g., Chinese, Japanese, Korean)"]
    """
    if not isinstance(race_entry, str):
        return []
    
    races = []
    current_race = ""
    paren_depth = 0
    
    for char in race_entry:
        if char == '(':
            paren_depth += 1
            current_race += char
        elif char == ')':
            paren_depth -= 1
            current_race += char
        elif char == ',' and paren_depth == 0:
            # This comma is a separator, not inside parentheses
            if current_race.strip():
                races.append(current_race.strip())
            current_race = ""
        else:
            current_race += char
    
    # Add the last race
    if current_race.strip():
        races.append(current_race.strip())
    
    return races

if race_col in df_alumni.columns:
    print(f"Race column '{race_col}' found!")
    
    # Get actual data (skip row 0 which is the question text)
    df_data = df_alumni.iloc[1:].copy()
    
    print(f"\nTotal responses: {len(df_data)}")
    print(f"Responses with race data: {df_data[race_col].notna().sum()}")
    print(f"Missing race data: {df_data[race_col].isna().sum()}")
    # Parse all races and collect unique values
    all_races = []
    for race_entry in df_data[race_col].dropna():
        races = parse_races(race_entry)
        all_races.extend(races)
    
    # Get unique race values
    unique_races = sorted(set(all_races))
    
    print("\n" + "="*80)
    print(f"All unique race options (total: {len(unique_races)}):")
    print("="*80)
    for idx, race in enumerate(unique_races, 1):
        count = all_races.count(race)
        print(f"{idx}. {race} (n={count})")
    
else:
    print(f"Column '{race_col}' not found!")
    print("\nAvailable columns:")
    print(df_alumni.columns.tolist())

Race column 'Q29' found!

Total responses: 40
Responses with race data: 39
Missing race data: 1

All unique race options (total: 7):
1. A race that is not listed here (n=2)
2. American Indian or Alaska Native (n=1)
3. Black or African American (n=12)
4. East Asian (e.g., Chinese, Japanese, Korean) (n=4)
5. Native Hawaiian or Pacific Islander (n=1)
6. South Asian (e.g., Indian, Pakistani, Sri Lankan) (n=4)
7. White or Caucasian (n=18)


## Flag Underserved/Marginalized Races

Create a flag for respondents who identify with at least one underserved race. Each person is counted only once regardless of how many underserved races they selected.

In [4]:
# Define underserved/marginalized race categories
# Based on typical URM (Underrepresented Minority) definitions in CS/STEM
underserved_races = [
    'Black or African American',
    'American Indian or Alaska Native',
    'Native Hawaiian or Pacific Islander',
    'Middle Eastern (e.g., Egyptian, Saudi Arabian, Lebanese)',
    'A race that is not listed here',
]

# Get actual data (skip row 0 which is the question text)
df_data = df_alumni.iloc[1:].copy()

# Create a function to check if a person has any underserved race
def has_underserved_race(race_entry):
    """
    Check if the race entry contains at least one underserved race.
    Returns True if yes, False otherwise.
    """
    if pd.isna(race_entry):
        return False
    
    races = parse_races(race_entry)
    
    # Check if any of the person's races are in the underserved list
    for race in races:
        if race in underserved_races:
            return True
    return False

# Apply the flag to each row
df_data['underserved_flag'] = df_data[race_col].apply(has_underserved_race)

# Count unique people with underserved race
total_responses = len(df_data)
underserved_count = df_data['underserved_flag'].sum()
percentage = (underserved_count / total_responses * 100) if total_responses > 0 else 0

print("="*80)
print("UNDERSERVED RACE ANALYSIS")
print("="*80)
print(f"\nUnderserved race categories flagged:")
for idx, race in enumerate(underserved_races, 1):
    print(f"  {idx}. {race}")

print(f"\n{'─'*80}")
print(f"Total respondents: {total_responses}")
print(f"Respondents with at least one underserved race: {underserved_count}")
print(f"Percentage: {percentage:.1f}%")
print(f"{'─'*80}")

# Show breakdown by specific underserved race
print("\nBreakdown by specific underserved race:")
print("(Note: People may be counted in multiple categories if they selected multiple races)")
for race in underserved_races:
    count = sum(1 for entry in df_data[race_col].dropna() 
                if race in parse_races(entry))
    print(f"  {race}: {count} respondents")

UNDERSERVED RACE ANALYSIS

Underserved race categories flagged:
  1. Black or African American
  2. American Indian or Alaska Native
  3. Native Hawaiian or Pacific Islander
  4. Middle Eastern (e.g., Egyptian, Saudi Arabian, Lebanese)
  5. A race that is not listed here

────────────────────────────────────────────────────────────────────────────────
Total respondents: 40
Respondents with at least one underserved race: 16
Percentage: 40.0%
────────────────────────────────────────────────────────────────────────────────

Breakdown by specific underserved race:
(Note: People may be counted in multiple categories if they selected multiple races)
  Black or African American: 12 respondents
  American Indian or Alaska Native: 1 respondents
  Native Hawaiian or Pacific Islander: 1 respondents
  Middle Eastern (e.g., Egyptian, Saudi Arabian, Lebanese): 0 respondents
  A race that is not listed here: 2 respondents


## Add Flag to Main DataFrame and Export

In [5]:
# Add the underserved_race_flag column to the main dataframe
# Initialize the column with False for all rows
df_alumni['underserved_race_flag'] = False

# Apply the flag to actual data rows (skip row 0 which is the question text)
for idx in df_data.index:
    df_alumni.loc[idx, 'underserved_race_flag'] = df_data.loc[idx, 'underserved_flag']

# Export to CSV
output_file = 'CS_alumni_Survey_with_flags.csv'
df_alumni.to_csv(output_file, index=False)

print(f"✓ Added 'underserved_race_flag' column to dataframe")
print(f"✓ Exported data to: {output_file}")
print(f"\nDataframe shape: {df_alumni.shape}")
print(f"New column added: 'underserved_race_flag'")

# Show summary
print(f"\nFlag summary (excluding question text row):")
print(f"  True (has underserved race): {df_alumni.iloc[1:]['underserved_race_flag'].sum()}")
print(f"  False (no underserved race): {(~df_alumni.iloc[1:]['underserved_race_flag']).sum()}")

✓ Added 'underserved_race_flag' column to dataframe
✓ Exported data to: CS_alumni_Survey_with_flags.csv

Dataframe shape: (41, 52)
New column added: 'underserved_race_flag'

Flag summary (excluding question text row):
  True (has underserved race): 16
  False (no underserved race): 24
